# Member D: College production

**Tasks**

- [X] Get a CFBD key, put it in `.env`, run `python -m src.data.pull_college --check` for coverage by season.
- [X] Full pull, then zip `data/raw/college/` and upload to the team Drive.
- [X] Per position group: available stats, coverage, missingness by year and school level.
- [X] Write `reports/college_feature_audit.md`. Flag OL and other positions with almost no stats.

See `docs/week1_tasks.md` and the project plan for details. Reusable code goes in `src/`.

In [284]:
import sys; sys.path.append('..')
import pandas as pd
from src.utils import *

In [285]:
# this just gives me an error, so I'm guessing it will be built later
# master = pd.read_parquet(PROCESSED / 'master.parquet')
# master.head()

In [286]:
stats = pd.concat([pd.read_parquet(f) for f in sorted(RAW_COLLEGE.glob("player_stats_*.parquet"))], ignore_index=True)
picks = pd.concat([pd.read_parquet(f) for f in sorted(RAW_COLLEGE.glob("draft_picks_*.parquet"))], ignore_index=True)

stats.info()
stats.head()

<class 'pandas.DataFrame'>
RangeIndex: 1000365 entries, 0 to 1000364
Data columns (total 9 columns):
 #   Column      Non-Null Count    Dtype
---  ------      --------------    -----
 0   season      1000365 non-null  int64
 1   playerId    1000365 non-null  str  
 2   player      1000365 non-null  str  
 3   position    1000365 non-null  str  
 4   team        1000365 non-null  str  
 5   conference  1000365 non-null  str  
 6   category    1000365 non-null  str  
 7   statType    1000365 non-null  str  
 8   stat        1000365 non-null  str  
dtypes: int64(1), str(8)
memory usage: 119.8 MB


,season,playerId,player,position,team,conference,category,statType,stat
0,2008,27200,Michael Johnson,WR,UNLV,Mountain West,puntReturns,AVG,1.5
1,2008,27200,Michael Johnson,WR,UNLV,Mountain West,puntReturns,LONG,16
2,2008,27200,Michael Johnson,WR,UNLV,Mountain West,puntReturns,NO,6
3,2008,27200,Michael Johnson,WR,UNLV,Mountain West,puntReturns,TD,0
4,2008,27200,Michael Johnson,WR,UNLV,Mountain West,puntReturns,YDS,9


In [287]:
picks.info()
picks.head()

<class 'pandas.DataFrame'>
RangeIndex: 3828 entries, 0 to 3827
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   collegeAthleteId         3761 non-null   float64
 1   nflAthleteId             3828 non-null   int64  
 2   collegeId                3828 non-null   int64  
 3   collegeTeam              3828 non-null   str    
 4   collegeConference        3777 non-null   str    
 5   nflTeamId                3828 non-null   int64  
 6   nflTeam                  3828 non-null   str    
 7   year                     3828 non-null   int64  
 8   overall                  3828 non-null   int64  
 9   round                    3828 non-null   int64  
 10  pick                     3828 non-null   int64  
 11  name                     3828 non-null   str    
 12  position                 3828 non-null   str    
 13  height                   3777 non-null   float64
 14  weight                   3798 non-n

,collegeAthleteId,nflAthleteId,collegeId,collegeTeam,collegeConference,nflTeamId,nflTeam,year,overall,round,pick,name,position,height,weight,preDraftRanking,preDraftPositionRanking,preDraftGrade,hometownInfo
0,2130748.0,12044,130,Michigan,Big Ten,15,Miami,2008,1,1,1,Jake Long,Offensive Tackle,79.0,313.0,4.0,1.0,98.0,"{'city': None, 'country': None, 'countyFips': ..."
1,2155169.0,11845,258,Virginia,ACC,14,Los Angeles,2008,2,1,2,Chris Long,Defensive End,75.0,272.0,1.0,1.0,98.0,"{'city': None, 'country': None, 'countyFips': ..."
2,2125735.0,11586,103,Boston College,ACC,1,Atlanta,2008,3,1,3,Matt Ryan,Quarterback,77.0,228.0,5.0,1.0,98.0,"{'city': None, 'country': None, 'countyFips': ..."
3,2180249.0,12135,8,Arkansas,SEC,13,Las Vegas,2008,4,1,4,Darren McFadden,Running Back,73.0,211.0,3.0,1.0,98.0,"{'city': None, 'country': None, 'countyFips': ..."
4,2156209.0,11996,99,LSU,SEC,12,Kansas City,2008,5,1,5,Glenn Dorsey,Defensive Tackle,74.0,297.0,2.0,1.0,98.0,"{'city': None, 'country': None, 'countyFips': ..."


In [288]:
picks["pos_group"] = picks["position"].map(POSITION_MAP)   # rename "position" if the column is called something else
picks["pos_group"].value_counts(dropna=False)

pos_group
NaN    3828
Name: count, dtype: int64

In [289]:
print(stats["season"].agg(["min", "max"]), picks["year"].agg(["min", "max"]))

min    2008
max    2022
Name: season, dtype: int64 min    2008
max    2022
Name: year, dtype: int64


Everything looks to be accounted for, so now I'll start working with the data.

In [290]:
import numpy as np

### Cleaning Up the Picks

In [291]:
# --- Clean picks (reloads from disk, safe to rerun) ---
picks = pd.concat([pd.read_parquet(f) for f in sorted(RAW_COLLEGE.glob("draft_picks_*.parquet"))], ignore_index=True)

FULL = {"Quarterback":"QB","Running Back":"RB","Fullback":"FB","Wide Receiver":"WR","Tight End":"TE",
        "Offensive Tackle":"OT","Guard":"G","Offensive Guard":"G","Center":"C","Offensive Line":"OT",
        "Defensive End":"DE","Outside Linebacker":"OLB","Defensive Tackle":"DT","Nose Tackle":"NT",
        "Linebacker":"LB","Inside Linebacker":"ILB","Cornerback":"CB","Safety":"S","Free Safety":"FS",
        "Strong Safety":"SS","Kicker":"K","Place Kicker":"K","Punter":"P","Long Snapper":"LS",
        "Kick Returner":"KR","Defensive Back":"DB"}

picks = picks[picks["year"].between(CLASS_MIN, CLASS_MAX)].copy()
picks["pos_abbr"] = picks["position"].replace(FULL)
picks["pos_group"] = picks["pos_abbr"].map(POSITION_MAP)
picks["pid"] = picks["collegeAthleteId"].astype("Int64").astype(str)

# Defensive Backs → CB or S from college position (default CB)
col_pos = stats.assign(pid=stats["playerId"].astype(str)).groupby("pid")["position"].agg(lambda s: s.mode().iat[0])
db = picks["pos_abbr"] == "DB"
picks.loc[db, "pos_group"] = picks.loc[db, "pid"].map(col_pos).map(POSITION_MAP).fillna("CB")
print(f"{db.sum()} Defensive Backs →", picks.loc[db, "pos_group"].value_counts().to_dict())

excluded = ["K", "P", "LS", "KR", "PR"]
print("Unmapped:", picks.loc[picks["pos_group"].isna() & ~picks["pos_abbr"].isin(excluded), "position"].unique())
picks = picks.dropna(subset=["pos_group"])

# School level
POWER = {"ACC","Big 12","Big Ten","SEC","Pac-12","Pac-10","Big East"}
G5 = {"American Athletic","Mountain West","Mid-American","Sun Belt","Conference USA","FBS Independents","Western Athletic"}
picks["level"] = np.select([picks["collegeConference"].isin(POWER), picks["collegeConference"].isin(G5)],
                           ["Power", "G5/Ind"], "FCS/Other")
print(picks["pos_group"].value_counts())


3 Defensive Backs → {'CB': 3}
Unmapped: <ArrowStringArray>
[]
Length: 0, dtype: str
pos_group
EDGE    547
CB      420
WR      412
RB      293
IOL     285
IDL     273
OT      263
S       261
TE      182
LB      177
QB      150
Name: count, dtype: int64


Two things to note:
1. All of the Outside Linebackers (`OLB`) were mapped to Edge Rushers (`EDGE`), not Linebackers (`LB`). This is why there are so many more Edge Rushers (547) than Linebackers (177). This is according to `POSITION_MAP` in `src/utils`.
2. There were 3 players listed as Defensive Backs. My code tried to match them to either Safety (`S`) or Corner (`CB`), with Corner as the default option if no information was available. I'll confirm by hand rather than assuming.


In [292]:
db_rows = picks[picks["pos_abbr"] == "DB"][["name", "year", "collegeTeam", "pid", "weight"]]
db_rows["college_pos"] = db_rows["pid"].map(col_pos)
db_rows

,name,year,collegeTeam,pid,weight,college_pos
978,Mikail Baker,2011,Baylor,173601,NaN,DB
981,Malcolm Williams,2011,TCU,-1033734,NaN,NaN
997,Jimmy Wilson,2011,Montana,2160152,NaN,NaN


After checking online, Mikail Baker and Malcolm Williams played CB, while Jimmy Wilson played Safety.

In [293]:
DB_OVERRIDES = {
    "Mikail Baker": "CB",
    "Malcolm Williams": "CB",
    "Jimmy Wilson": "S",
}
for name, grp in DB_OVERRIDES.items():
    mask = (picks["name"] == name) & (picks["pos_abbr"] == "DB")
    assert mask.sum() == 1, f"{name}: matched {mask.sum()} rows, check spelling"
    picks.loc[mask, "pos_group"] = grp

In [294]:
picks.loc[picks["pos_abbr"] == "DB", ["name", "year", "collegeTeam", "pos_group"]]

,name,year,collegeTeam,pos_group
978,Mikail Baker,2011,Baylor,CB
981,Malcolm Williams,2011,TCU,CB
997,Jimmy Wilson,2011,Montana,S


Fixed! Next is...
### Cleaning up the Player Stats

In [295]:
# --- Clean stats ---
stats["pid"] = stats["playerId"].astype(str)
stats["value"] = pd.to_numeric(stats["stat"], errors="coerce")
stats["stat_name"] = stats["category"] + "." + stats["statType"]

d = stats.merge(picks[["pid", "year", "pos_group", "level"]], on="pid", how="inner")
d = d[d["season"] < d["year"]]                  # pre-draft seasons only
final = d[d["season"] == d["year"] - 1]

print("Non-numeric stats:", stats["value"].isna().sum())
print("Drafted players matched to any stats:", d["pid"].nunique(), "of", len(picks))
print("Matched in final season:", final["pid"].nunique())

Non-numeric stats: 0
Drafted players matched to any stats: 2343 of 3263
Matched in final season: 2142


There are no non-numeric stats, which makes my life easier, and a surprisingly high amount of picks matched to their college stats, given that most linemen don't record stats.

In [296]:
ol = d[d["pos_group"].isin(["OT", "IOL"])]
print(ol.groupby("stat_name")["value"].agg(rows="size", nonzero=lambda v: (v > 0).sum(), total="sum")
        .sort_values("nonzero", ascending=False))

                     rows  nonzero  total
stat_name                                
defensive.TOT         157      154  189.0
defensive.SOLO        157      128  149.0
receiving.REC          31       26   65.0
rushing.YDS            33       23  142.0
rushing.LONG           33       22  134.0
receiving.LONG         31       21  161.0
receiving.YDS          31       21  454.0
rushing.CAR            33       21   22.0
receiving.YPR          31       16   65.6
rushing.YPC            33       16   77.5
rushing.TD             33       15   16.0
fumbles.FUM            10        8    8.0
receiving.TD           31        4    4.0
kickReturns.AVG         3        3   26.0
kickReturns.NO          3        3    3.0
kickReturns.YDS         3        3   26.0
defensive.QB HUR      157        3    3.0
defensive.TFL         157        3    3.0
kickReturns.LONG        3        3   26.0
fumbles.REC            10        2    2.0
fumbles.LOST           10        2    2.0
passing.ATT             2        2

Some linemen have defensive stats, likely from a tackle after an interception. Overall, this is not common and doesn't measure OL quality.

In [297]:
dn = d[d["value"] > 0]                                   # drop zero-value placeholder rows
picks["has_any"]   = picks["pid"].isin(dn["pid"])
picks["has_final"] = picks["pid"].isin(dn.loc[dn["season"] == dn["year"] - 1, "pid"])

coverage = (picks.groupby("pos_group")
            .agg(drafted=("pid", "size"), pct_any=("has_any", "mean"), pct_final=("has_final", "mean"))
            .reindex(POSITION_GROUPS).round(3))
coverage

,drafted,pct_any,pct_final
pos_group,,,
QB,150,0.933,0.927
RB,293,0.925,0.918
WR,412,0.908,0.893
TE,182,0.874,0.868
OT,263,0.281,0.156
IOL,285,0.302,0.151
EDGE,547,0.651,0.576
IDL,273,0.549,0.476
LB,177,0.927,0.853


If a player has any recorded college stats, they will be represented under `pct_any`. Meanwhile, `pct_final` is for players who have recorded college stats specifically in their last season before getting drafted. `pct_final` is slightly lower than `pct_any` because of injuries.

In [298]:
RELEVANT = {
    "QB":   ["passing.ATT", "passing.YDS", "passing.TD", "passing.INT", "rushing.YDS"],
    "RB":   ["rushing.CAR", "rushing.YDS", "rushing.TD", "receiving.REC", "receiving.YDS"],
    "WR":   ["receiving.REC", "receiving.YDS", "receiving.TD"],
    "TE":   ["receiving.REC", "receiving.YDS", "receiving.TD"],
    "OT":   [],
    "IOL":  [],
    "EDGE": ["defensive.SACKS", "defensive.TFL", "defensive.QB HUR", "defensive.TOT"],
    "IDL":  ["defensive.SACKS", "defensive.TFL", "defensive.QB HUR", "defensive.TOT"],
    "LB":   ["defensive.TOT", "defensive.SOLO", "defensive.TFL", "defensive.SACKS", "defensive.PD", "interceptions.INT"],
    "CB":   ["defensive.PD", "interceptions.INT", "defensive.TOT", "defensive.SOLO"],
    "S":    ["defensive.PD", "interceptions.INT", "defensive.TOT", "defensive.SOLO"],
}

Only including relevant stats for each position*

In [299]:
# Sanity check: every listed stat name actually exists in the data
all_names = set(d["stat_name"])
print("Missing stat names:", {s for v in RELEVANT.values() for s in v} - all_names)

Missing stat names: set()


Nothing missing!

In [300]:
# keeping only stats that are relevant to the players' positions
rel_pairs = pd.DataFrame([(g, s) for g, v in RELEVANT.items() for s in v], columns=["pos_group", "stat_name"])
dr = dn.merge(rel_pairs, on=["pos_group", "stat_name"])

In [301]:
picks["has_rel_any"]   = picks["pid"].isin(dr["pid"])
picks["has_rel_final"] = picks["pid"].isin(dr.loc[dr["season"] == dr["year"] - 1, "pid"])

In [302]:
coverage = (picks.groupby("pos_group")
            .agg(drafted=("pid", "size"),
                 pct_any=("has_any", "mean"),           # naive, from previous cell
                 rel_any=("has_rel_any", "mean"),
                 rel_final=("has_rel_final", "mean"))
            .reindex(POSITION_GROUPS).round(3))
coverage

,drafted,pct_any,rel_any,rel_final
pos_group,,,,
QB,150,0.933,0.933,0.927
RB,293,0.925,0.925,0.918
WR,412,0.908,0.908,0.891
TE,182,0.874,0.874,0.868
OT,263,0.281,0.000,0.000
IOL,285,0.302,0.000,0.000
EDGE,547,0.651,0.389,0.384
IDL,273,0.549,0.432,0.418
LB,177,0.927,0.927,0.842


There is a drop to 0 for the `OT` and `IOT` stats, which makes sense. However, there is also a noticeable drop for `LB`, of almost 40 percentage points. I'll look into this further.

In [303]:
# defenders who have nonzero stats but none of the relevant ones: what do they have?
gap = picks[picks["has_any"] & ~picks["has_rel_any"] & picks["pos_group"].isin(["LB", "EDGE", "IDL"])]
print(len(gap), "players")
print(dn[dn["pid"].isin(gap["pid"])]["stat_name"].value_counts().head(15))

175 players
stat_name
interceptions.INT    227
interceptions.AVG    183
interceptions.YDS    183
interceptions.TD      67
puntReturns.NO        30
puntReturns.AVG       27
puntReturns.LONG      27
puntReturns.YDS       27
receiving.LONG        20
receiving.REC         20
receiving.YDS         20
receiving.YPR         20
kickReturns.NO        20
rushing.CAR           19
rushing.LONG          18
Name: count, dtype: int64


The biggest stat to include is interceptions, as they are significant on defense, for `LB` and even `EDGE`. The other special teams and offensive stats, meanwhile, do not add much.

In [304]:
# spot-check a few: name/position in picks vs. what CFBD stats say
check = gap[["pid", "name", "pos_group", "collegeTeam", "year"]].head(10).merge(
    stats[["pid", "player", "position", "team"]].drop_duplicates("pid"), on="pid", how="left")
check

,pid,name,pos_group,collegeTeam,year,player,position,team
0,175300,Ndamukong Suh,IDL,Nebraska,2010,Ndamukong Suh,DT,Nebraska
1,191442,Brandon Graham,EDGE,Michigan,2010,Brandon Graham,DE,Michigan
2,482895,Jason Pierre-Paul,EDGE,South Florida,2010,Jason Pierre-Paul,DE,South Florida
3,191607,Sean Weatherspoon,EDGE,Missouri,2010,Sean Weatherspoon,LB,Missouri
4,238495,Linval Joseph,IDL,East Carolina,2010,Linval Joseph,DL,East Carolina
5,189938,Daryl Washington,EDGE,TCU,2010,Daryl Washington,LB,TCU
6,188306,Thaddeus Gibson,EDGE,Ohio State,2010,Thaddeus Gibson,LB,Ohio State
7,188288,A.J. Edds,EDGE,Iowa,2010,A.J. Edds,LB,Iowa
8,174556,Keenan Clayton,EDGE,Oklahoma,2010,Keenan Clayton,LB,Oklahoma
9,190688,Eric Norwood,EDGE,South Carolina,2010,Eric Norwood,LB,South Carolina


This is interesting becuase there are notable players with productive seasons on this list, yet the CFBD dataset doesn't show any recorded statistics. These all seem to be in the early seasons, so I'll look into if that is the trend/issue.

In [305]:
# share of all defensive players (not just drafted) with any defensive.* stat, by season
def_pos = ["DL", "DT", "DE", "LB", "DB", "CB", "S"]
s = stats[stats["position"].isin(def_pos)]
has_def = s[s["category"] == "defensive"].groupby("season")["pid"].nunique()
all_def = s.groupby("season")["pid"].nunique()
(has_def / all_def).round(2)

season
2008     NaN
2009     NaN
2010     NaN
2011     NaN
2012     NaN
2013     NaN
2014     NaN
2015     NaN
2016    0.99
2017    0.99
2018    0.97
2019    0.99
2020    0.98
2021    0.96
2022    0.90
Name: pid, dtype: float64

It looks like there was no defensive category up until 2016. This explains why players in 2010 like Suh had interceptions but no sacks or tackles. But before accepting this, I'll confirm.

In [306]:
for yr in [2012, 2015, 2016]:
    print(yr, sorted(stats.loc[stats["season"] == yr, "category"].unique()))

2012 ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
2015 ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
2016 ['defensive', 'fumbles', 'interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']


Yup, it looks like there are no fumbles or defensive stats before 2016. One last thing I can do is call the API to see if the pull script missed it.

In [307]:
import os, requests
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

r = requests.get("https://api.collegefootballdata.com/stats/player/season",
                 params={"year": 2013, "category": "defensive"},
                 headers={"Authorization": f"Bearer {os.environ['CFBD_API_KEY']}"})
print(r.status_code, len(r.json()))
print(r.json()[:3])

200 0
[]


Yup, it came up empty.

In [308]:
r = requests.get("https://api.collegefootballdata.com/games/players",
                 params={"year": 2013, "week": 5},
                 headers={"Authorization": f"Bearer {os.environ['CFBD_API_KEY']}"})
games = r.json()
cats = {c["name"] for g in games for t in g["teams"] for c in t["categories"]}
print(len(games), sorted(cats))

49 ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']


And after checking the box score, still no defensive or fumble statistics.

This is very important. CFBD has no defensive stats (tackles, sacks, TFL, hurries) before the 2016 season, even after directly confirming with the API. The only defensive statistic available pre-2016 is interceptions. That means most defenders in the 2010–2016 classes have no college defensive production that we can measure. 

This is not an issue with the offense, though (offense covers all of 2010–2022).

Options: 
1. Train defensive Layer 2 on the 2017–22 classes only, 
2. Use combine and pick slot only for earlier defenders, 
3. Add another source for 2008–15 (which means a scraper and name matching). 

Which do we want?

In [309]:
# tag each drafted defender with an era and whether they have any defensive.* stat
DEF_GROUPS = ["EDGE", "IDL", "LB", "CB", "S"]
has_def = dn.loc[dn["category"] == "defensive", "pid"]
era = (picks[picks["pos_group"].isin(DEF_GROUPS)]
       .assign(era=lambda x: np.where(x["year"] <= 2016, "2010-16", "2017-22"),
               has_def_any=lambda x: x["pid"].isin(has_def)))

Splits drafted defenders into the 2010–16 classes (final season before CFBD added defensive stats) and the 2017–22 classes, and flags who has any tackle/sack/TFL-type stat.

In [310]:
era_table = (era.groupby(["pos_group", "era"])
                .agg(drafted=("pid", "size"),
                     rel_final=("has_rel_final", "mean"),
                     def_any=("has_def_any", "mean"))
                .unstack("era")
                .reindex(DEF_GROUPS).round(3))
era_table

drafted         rel_final         def_any        
era       2010-16 2017-22   2010-16 2017-22 2010-16 2017-22
pos_group                                                  
EDGE          317     230     0.000   0.913     0.0   0.926
IDL           148     125     0.000   0.912     0.0   0.944
LB             79      98     0.696   0.959     0.0   0.980
CB            224     195     0.643   0.892     0.0   0.928
S             134     128     0.612   0.922     0.0   0.961

Era table: final-season relevant coverage (`rel_final`) and share with any `defensive.*` stat (`def_any`), before vs. after 2016.

`def_any` is 0 for every position in 2010–16, which confirms the gap. EDGE and IDL drop to 0 coverage there. LB, CB, and S keep 61–70% only because of interceptions, so for those early classes "covered" means "we know their INT count and nothing else." From 2017 on, every defensive group is about 89–96% covered.

In [311]:
# share of players in each position group with a nonzero value for each stat
# 2017-22 classes only, so the pre-2016 defensive gap doesn't hide which stats exist
p = picks[(picks["year"] >= 2017) & picks["has_any"]]
base = p.groupby("pos_group")["pid"].nunique()
avail = (dn[dn["pid"].isin(p["pid"])]
         .groupby(["pos_group", "stat_name"])["pid"].nunique()
         .div(base, level="pos_group").rename("share").reset_index())

Builds `avail`: for each position group and stat, the share of drafted players (2017–22 classes, with any stats) who have a nonzero value in a pre-draft season.

In [312]:
# how available is each stat we picked in RELEVANT? flag anything under 10%
rel_check = rel_pairs.merge(avail, how="left").fillna({"share": 0})
rel_check["rare"] = rel_check["share"] < 0.10
rel_check.round(3)

,pos_group,stat_name,share,rare
0,QB,passing.ATT,1.000,False
1,QB,passing.YDS,1.000,False
2,QB,passing.TD,1.000,False
3,QB,passing.INT,1.000,False
4,QB,rushing.YDS,0.905,False
5,RB,rushing.CAR,1.000,False
6,RB,rushing.YDS,1.000,False
7,RB,rushing.TD,0.985,False
8,RB,receiving.REC,1.000,False
9,RB,receiving.YDS,1.000,False


Checks every stat in `RELEVANT` against `avail`.

None are rare: every chosen stat is present for at least 90% of the position, except LB interceptions (78%). That makes sense, since many linebackers never record one.

In [313]:
# stats NOT in RELEVANT that are common for the position: top 5 per group
others = (avail.merge(rel_pairs, how="left", indicator=True)
               .query("_merge == 'left_only'").drop(columns="_merge")
               .sort_values(["pos_group", "share"], ascending=[True, False])
               .groupby("pos_group").head(5))
others.round(3)

,pos_group,stat_name,share
5,CB,defensive.TFL,0.929
10,CB,interceptions.AVG,0.780
13,CB,interceptions.YDS,0.780
9,CB,fumbles.REC,0.451
2,CB,defensive.SACKS,0.385
42,EDGE,defensive.SOLO,1.000
39,EDGE,defensive.PD,0.883
48,EDGE,fumbles.REC,0.718
50,EDGE,interceptions.INT,0.390
49,EDGE,interceptions.AVG,0.310


Shows the five most common stats each position has that aren't in `RELEVANT`, to catch anything I left out.

A few look worth adding when I build the candidate features: QB `rushing.CAR`, LB `defensive.QB HUR`, and CB/S `defensive.TFL`. Rate stats like `passing.PCT`, `passing.YPA`, and `rushing.YPC` can be computed from the counts, so they don't need to be pulled separately. `defensive.TOT` showing up for offensive players and OL is just post-turnover tackles (same as the OL check earlier), so I'm ignoring it.

In [314]:
# missing = no nonzero relevant stat in any pre-draft season (OT/IOL left out: 100% by definition)
SKILL = [g for g in POSITION_GROUPS if RELEVANT[g]]
m = picks[picks["pos_group"].isin(SKILL)].assign(missing=lambda x: ~x["has_rel_any"])

Builds `m`: one row per drafted player (OT/IOL excluded), with `missing = True` if they have no nonzero relevant stat in any pre-draft season.

In [315]:
miss_year = (m.pivot_table(index="pos_group", columns="year", values="missing", aggfunc="mean")
               .reindex(SKILL).round(2))
miss_year

year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
pos_group,,,,,,,,,,,,,
QB,0.14,0.17,0.09,0.09,0.00,0.00,0.07,0.10,0.08,0.00,0.00,0.10,0.00
RB,0.20,0.17,0.14,0.12,0.00,0.14,0.09,0.07,0.05,0.00,0.00,0.00,0.00
WR,0.19,0.18,0.12,0.14,0.12,0.14,0.12,0.09,0.03,0.00,0.00,0.03,0.07
TE,0.30,0.25,0.09,0.00,0.00,0.16,0.44,0.21,0.00,0.00,0.08,0.09,0.06
EDGE,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.21,0.09,0.05,0.03,0.03,0.03
IDL,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.09,0.19,0.00,0.00,0.00,0.05
LB,0.15,0.17,0.11,0.09,0.12,0.15,0.15,0.08,0.00,0.07,0.00,0.00,0.00
CB,0.32,0.34,0.27,0.17,0.33,0.07,0.31,0.09,0.06,0.10,0.00,0.00,0.14
S,0.21,0.39,0.40,0.22,0.24,0.25,0.19,0.08,0.06,0.00,0.04,0.00,0.05


Missingness by draft class (share of each position with no relevant stats).

EDGE and IDL are 100% missing for 2010–16 (the defensive gap), then drop to about 0–20%. Offense is missing 10–20% in the earliest classes and close to 0 after 2018. The 2010–11 classes are worse partly because their pre-draft history starts in 2008, when the raw files begin. TE swings a lot year to year because each class is small.

In [316]:
miss_level = (m.pivot_table(index="pos_group", columns="level", values="missing", aggfunc="mean")
                .reindex(SKILL).round(2))
n_level = m.pivot_table(index="pos_group", columns="level", values="pid", aggfunc="size").reindex(SKILL)
display(miss_level, n_level)

level,FCS/Other,G5/Ind,Power
pos_group,,,
QB,0.60,0.03,0.03
RB,0.50,0.04,0.06
WR,0.77,0.05,0.03
TE,0.85,0.12,0.06
EDGE,0.85,0.57,0.59
IDL,0.88,0.58,0.53
LB,0.75,0.00,0.07
CB,0.81,0.10,0.09
S,0.86,0.14,0.09


level,FCS/Other,G5/Ind,Power
pos_group,,,
QB,10,33,107
RB,14,53,226
WR,31,91,290
TE,13,42,127
EDGE,48,98,401
IDL,24,38,211
LB,4,28,145
CB,48,87,284
S,21,44,197


Missingness by school level, with the player counts underneath.

FCS/Other is the clear problem: 50–88% missing at every position, likely because CFBD's season stats mostly cover FBS teams. Power and G5 are close to each other on offense (3–12%). For defense, they're both around 55–60% for EDGE/IDL, which is the pre-2016 gap again, not school level. FCS counts are small (4 LBs, 10 QBs), so those rates are noisy.

In [317]:
# picks with no stats at all: unlinked CFBD id (negative) vs. stats filed under a different id
no_stats = picks[~picks["has_any"]]
unlinked = no_stats["pid"].str.startswith("-")
names = stats[["player", "team", "season"]].drop_duplicates()
by_name = (no_stats[~unlinked]
           .merge(names, left_on=["name", "collegeTeam"], right_on=["player", "team"])
           .query("season < year"))
print(f"{len(no_stats)} picks with no stats: {unlinked.sum()} have an unlinked (negative) id, "
      f"{by_name['pid'].nunique()} have stats under a different id (exact name + school)")
by_name.drop_duplicates("pid")["year"].value_counts().sort_index()

920 picks with no stats: 465 have an unlinked (negative) id, 44 have stats under a different id (exact name + school)


year
2010    20
2011     5
2012     2
2013     4
2015     3
2016     5
2017     2
2018     3
Name: count, dtype: int64

Breaks down the 920 picks with no stats at all.

465 have a negative `collegeAthleteId`, which is how CFBD marks a pick that isn't linked to a college player record, so the ID join can never find them. Another 44 do have stats, but filed under a different `playerId` (for example, Rob Gronkowski and Jermaine Gresham), and most of those are in the 2010 class. So the direct ID join is clean when it hits, but it misses some players. I'm leaving these unmatched here and flagging it for the master-table matching instead.

In [318]:
# one headline stat per position, from the final pre-draft season
# uses `final` (keeps real zeros: a row exists only if the player has that category that season)
KEY_STAT = {"QB": "passing.YDS", "RB": "rushing.YDS", "WR": "receiving.YDS", "TE": "receiving.YDS",
            "EDGE": "defensive.SACKS", "IDL": "defensive.TFL", "LB": "defensive.TOT",
            "CB": "defensive.PD", "S": "defensive.TOT"}
key_pairs = pd.DataFrame(KEY_STAT.items(), columns=["pos_group", "stat_name"])
fk = final.merge(key_pairs, on=["pos_group", "stat_name"])

Builds `fk`: one row per player with the final-season value of their position's headline stat (`KEY_STAT`).

I used `final` instead of `dn` on purpose. A 0 here is a real zero (for example, an EDGE with tackles but no sacks), not a placeholder, because the row only exists if the player has stats in that category.

In [319]:
fk.groupby("pos_group")["value"].describe().reindex(list(KEY_STAT)).round(1)

,count,mean,std,min,25%,50%,75%,max
pos_group,,,,,,,,
QB,139.0,3307.3,973.3,156.0,2847.5,3316.0,3894.5,5967.0
RB,263.0,1010.0,547.6,-5.0,647.0,1060.0,1398.0,2587.0
WR,367.0,917.0,375.5,-3.0,675.5,913.0,1180.0,1856.0
TE,158.0,456.7,243.2,19.0,278.0,450.0,604.0,1352.0
EDGE,210.0,6.1,3.6,0.0,3.5,5.5,8.5,17.0
IDL,114.0,8.2,4.2,1.0,5.0,7.8,11.4,19.5
LB,94.0,84.8,31.8,11.0,65.2,85.0,103.8,181.0
CB,174.0,7.2,4.3,0.0,4.0,7.0,10.0,19.0
S,118.0,62.9,21.0,6.0,50.2,64.5,74.8,118.0


Summary of each headline stat (count, mean, spread, min/max).

The ranges look like real football: the median drafted QB threw about 3,300 yards, the median EDGE had 5.5 sacks. The small negative minimums for RB/WR are real (a few carries for a loss), not errors.

In [320]:
# the two highest values per position: are the outliers real seasons or data errors?
(fk.sort_values("value", ascending=False).groupby("pos_group").head(2)
   [["pos_group", "player", "season", "team", "stat_name", "value"]].sort_values("pos_group"))

,pos_group,player,season,team,stat_name,value
635,CB,Ahkello Witherspoon,2016,Colorado,defensive.PD,19.0
800,CB,Joshua Jackson,2017,Iowa,defensive.PD,18.0
949,EDGE,Josh Allen,2018,Kentucky,defensive.SACKS,17.0
1220,EDGE,Chase Young,2019,Ohio State,defensive.SACKS,16.5
1040,IDL,Quinnen Williams,2018,Alabama,defensive.TFL,19.5
1243,IDL,James Lynch,2019,Baylor,defensive.TFL,19.5
1142,LB,Evan Weaver,2019,California,defensive.TOT,181.0
932,LB,Ben Burr-Kirven,2018,Washington,defensive.TOT,165.0
1513,QB,Bailey Zappe,2021,Western Kentucky,passing.YDS,5967.0
1087,QB,Joe Burrow,2019,LSU,passing.YDS,5671.0


Lists the top two seasons per position to check the outliers.

They're all real, well-known seasons (Melvin Gordon's 2,587 rushing yards, Joe Burrow's 2019, Chase Young's 16.5 sacks), so there's nothing to clean.

In [321]:
lvl_median = fk.pivot_table(index="pos_group", columns="level", values="value", aggfunc="median").reindex(list(KEY_STAT))
lvl_n = fk.pivot_table(index="pos_group", columns="level", values="value", aggfunc="size").reindex(list(KEY_STAT))
display(lvl_median, lvl_n)

level,FCS/Other,G5/Ind,Power
pos_group,,,
QB,224.0,3554.5,3187.00
RB,125.0,1133.0,1022.00
WR,138.0,1117.0,866.00
TE,136.0,515.0,419.00
EDGE,0.0,6.5,5.75
IDL,1.0,10.5,7.50
LB,11.0,92.0,84.00
CB,0.0,8.5,7.00
S,7.0,67.0,65.00


level,FCS/Other,G5/Ind,Power
pos_group,,,
QB,3,32,104
RB,7,48,208
WR,6,85,276
TE,1,37,120
EDGE,6,42,162
IDL,3,16,95
LB,1,18,75
CB,7,40,127
S,2,17,99


Median headline stat by school level, with player counts underneath.

Drafted G5 players have higher raw numbers than Power players (WR 1,117 vs. 866 yards, QB 3,555 vs. 3,187). That's likely because a G5 player has to dominate weaker opponents to get drafted, so raw counts aren't comparable across levels. FCS medians are tiny (QB 224 yards, EDGE 0 sacks), which suggests CFBD only has some of their games. With 1–7 FCS players per position, I'd treat FCS college stats as missing instead of trusting them.

In [322]:
season_median = fk.pivot_table(index="pos_group", columns="season", values="value", aggfunc="median").reindex(list(KEY_STAT))
season_median

season,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
pos_group,,,,,,,,,,,,,
QB,2847.5,2995.5,3630.5,3437.5,3554.5,3855.0,3488.0,2946.0,3688.5,3555.00,2951.0,2830.0,3188.0
RB,1212.0,1014.0,1259.0,1075.0,1095.0,979.0,1070.0,1149.5,1169.5,985.00,1117.0,858.0,956.5
WR,927.0,899.5,956.0,874.0,1060.0,863.0,926.0,968.0,879.0,931.50,902.5,625.0,1003.0
TE,493.0,358.0,499.5,466.5,478.5,293.0,438.0,651.0,477.0,472.00,504.0,294.0,447.0
EDGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,4.5,6.50,5.5,5.0,6.0
IDL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,7.5,9.75,8.5,4.5,6.5
LB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96.0,85.5,82.00,84.5,81.0,81.0
CB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,9.0,9.00,8.0,4.0,6.0
S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71.0,76.5,65.00,68.5,42.0,60.0


Median headline stat by final season, to look for drift over time.

Most positions are fairly stable year to year, with noise from small groups (about 9–40 players per cell). The clear exception is 2020, the COVID-shortened season: WR (625), S (42), and CB (4.0) all drop sharply, because players got fewer games. Since CFBD has no games-played column, we can't convert to per-game rates. So I'd recommend standardizing each stat within season and position (for example, a percentile or z-score among that year's drafted players at the position), and adding school level as its own feature.

## Candidate features

In [323]:
# NCAA passer efficiency from final-season counts; under 100 attempts counts as missing (too noisy)
qb = (final[(final["pos_group"] == "QB") & (final["category"] == "passing")]
      .pivot_table(index=["pid", "player", "team", "season", "level"], columns="statType", values="value")
      .reset_index())
qb["pass_eff"] = (8.4*qb["YDS"] + 330*qb["TD"] + 100*qb["COMPLETIONS"] - 200*qb["INT"]) / qb["ATT"]
qb.loc[qb["ATT"] < 100, "pass_eff"] = np.nan

Computes each QB's final-season passer efficiency rating, the college formula, from attempts, completions, yards, TDs, and INTs. QBs with fewer than 100 attempts get `NaN`.

In [324]:
display(qb["pass_eff"].describe().round(1),
        qb.groupby("level")["pass_eff"].agg(["median", "count"]).round(1))

count    135.0
mean     152.2
std       18.9
min      105.9
25%      140.4
50%      150.5
75%      163.4
max      206.9
Name: pass_eff, dtype: float64

,median,count
level,,
FCS/Other,NaN,0
G5/Ind,152.2,32
Power,149.8,103


Distribution of passer efficiency, overall and by school level.

135 of 139 QBs have 100+ attempts. The median is about 150 and the top is Tua Tagovailoa's 2019 (206.9). Joe Burrow's 2019 comes out at 202.0, which matches his official rating, so the formula is right. Unlike raw passing yards, efficiency is nearly the same at G5 and Power (152 vs. 150), so a per-attempt stat is less tilted by level than a total.

In [325]:
# final-season relevant coverage by era, for every position group
cov = (picks.assign(era=np.where(picks["year"] <= 2016, "2010-16", "2017-22"))
            .pivot_table(index="pos_group", columns="era", values="has_rel_final", aggfunc="mean")
            .reindex(POSITION_GROUPS).round(2))

Builds `cov`: the share of each position with a relevant final-season stat, split by era, to sit next to the features below.

In [326]:
# candidate college features per position group (final season, plus career totals over pre-draft seasons)
# every group also gets: school level, combine + size, and pick slot (from other members' tables)
CANDIDATES = {
    "QB":   ("passing ATT/YDS/TD/INT, pass_eff (100+ att), rushing CAR/YDS", "career pass YDS/TD, seasons with stats"),
    "RB":   ("rushing CAR/YDS/TD, receiving REC/YDS, YPC",                    "career rush YDS, career REC"),
    "WR":   ("receiving REC/YDS/TD, YPR",                                      "career rec YDS/TD"),
    "TE":   ("receiving REC/YDS/TD, YPR",                                      "career rec YDS/TD"),
    "OT":   ("none",                                                           "none"),
    "IOL":  ("none",                                                           "none"),
    "EDGE": ("SACKS, TFL, QB HUR, TOT",                                        "career SACKS/TFL (2017+ classes)"),
    "IDL":  ("TFL, SACKS, QB HUR, TOT",                                        "career TFL/SACKS (2017+ classes)"),
    "LB":   ("TOT, SOLO, TFL, SACKS, QB HUR, PD, INT",                        "career TOT/TFL (2017+), career INT (all)"),
    "CB":   ("PD, INT, TOT, SOLO, TFL",                                        "career PD (2017+), career INT (all)"),
    "S":    ("PD, INT, TOT, SOLO, TFL",                                        "career TOT/PD (2017+), career INT (all)"),
}
CONCERNS = {
    "QB":   "small group (~150); efficiency noisy under 100 att",
    "RB":   "counting stats favor durable players",
    "WR":   "counting stats favor durable players",
    "TE":   "small classes; blocking TEs have few catches",
    "OT":   "no college production; size, combine, pick slot only",
    "IOL":  "no college production; size, combine, pick slot only",
    "EDGE": "no defensive stats before 2016; OLB->EDGE mapping inflates group",
    "IDL":  "no defensive stats before 2016",
    "LB":   "pre-2016 classes covered by INTs only",
    "CB":   "pre-2016 classes covered by INTs only",
    "S":    "pre-2016 classes covered by INTs only",
}
features = (pd.DataFrame.from_dict(CANDIDATES, orient="index", columns=["final_season", "career"])
              .assign(concerns=pd.Series(CONCERNS))
              .join(cov.add_prefix("rel_final_")))
features

,final_season,career,concerns,rel_final_2010-16,rel_final_2017-22
QB,"passing ATT/YDS/TD/INT, pass_eff (100+ att), r...","career pass YDS/TD, seasons with stats",small group (~150); efficiency noisy under 100...,0.92,0.94
RB,"rushing CAR/YDS/TD, receiving REC/YDS, YPC","career rush YDS, career REC",counting stats favor durable players,0.87,0.97
WR,"receiving REC/YDS/TD, YPR",career rec YDS/TD,counting stats favor durable players,0.84,0.94
TE,"receiving REC/YDS/TD, YPR",career rec YDS/TD,small classes; blocking TEs have few catches,0.82,0.92
OT,none,none,"no college production; size, combine, pick slo...",0.00,0.00
IOL,none,none,"no college production; size, combine, pick slo...",0.00,0.00
EDGE,"SACKS, TFL, QB HUR, TOT",career SACKS/TFL (2017+ classes),no defensive stats before 2016; OLB->EDGE mapp...,0.00,0.91
IDL,"TFL, SACKS, QB HUR, TOT",career TFL/SACKS (2017+ classes),no defensive stats before 2016,0.00,0.91
LB,"TOT, SOLO, TFL, SACKS, QB HUR, PD, INT","career TOT/TFL (2017+), career INT (all)",pre-2016 classes covered by INTs only,0.70,0.96
CB,"PD, INT, TOT, SOLO, TFL","career PD (2017+), career INT (all)",pre-2016 classes covered by INTs only,0.64,0.89


The candidate feature table: final-season and career stats per position, with coverage by era and the main concern for each group.

Every position also gets school level, combine and size measurements, and pick slot, all standardized within season and position as recommended in step 5. OT and IOL have no usable college production, so their models rely on those shared inputs alone. For EDGE and IDL, college features only exist for the 2017–22 classes, so that's the team decision from earlier. Career totals are short for the 2010–11 classes, since the raw data starts with the 2008 season. These are candidates, to be tested against the pick-slot baseline in Layer 2, not a final list.